<a href="https://colab.research.google.com/github/TanujaMore27/Deep-Learning/blob/main/5_pytorch_nn_module_pipeline_with_dataset_and_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import make_classification
import torch

In [2]:
X,y = make_classification(
    n_features=2,
    n_samples=10,
    n_redundant=0,
    n_informative=2,
    n_classes=2,
    random_state=42
)

In [3]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [4]:
X.shape

(10, 2)

In [5]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [6]:
y.shape

(10,)

In [8]:
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y,dtype=torch.long)

/tmp/ipykernel_8839/3276321781.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)


In [9]:
from torch.utils.data import Dataset,DataLoader

In [10]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self,index):
    return self.features[index],self.labels[index]

In [11]:
dataset = CustomDataset(X,y)

In [12]:
len(dataset)

10

In [13]:
dataset[2]

(tensor([-2.8954,  1.9769]), tensor(0))

In [14]:
dataloader = DataLoader(dataset,batch_size=2,shuffle=False)

In [15]:
for batch_features, batch_labels in dataloader:

  print(batch_features)
  print(batch_labels)
  print("-"*50)

tensor([[ 1.0683, -0.9701],
        [-1.1402, -0.8388]])
tensor([1, 0])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-0.7206, -0.9606]])
tensor([0, 0])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.9382, -0.5430]])
tensor([0, 1])
--------------------------------------------------
tensor([[ 1.7273, -1.1858],
        [ 1.7774,  1.5116]])
tensor([1, 1])
--------------------------------------------------
tensor([[ 1.8997,  0.8344],
        [-0.5872, -1.9717]])
tensor([1, 0])
--------------------------------------------------


In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [5]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [7]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [9]:
X_train.shape

(455, 30)

In [10]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [11]:
X_train_tensor.shape

torch.Size([455, 30])

In [15]:
from torch.utils.data import Dataset,DataLoader

class Custom(Dataset):
  def __init__(self,features,labels):
    self.features=features
    self.labels=labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
    return self.features[idx],self.labels[idx]

In [16]:
train_dataset = Custom(X_train_tensor,y_train_tensor)
test_dataset = Custom(X_test_tensor,y_test_tensor)

In [17]:
train_dataset[0]

(tensor([-0.0210, -0.7599, -0.0842, -0.1227, -0.8398, -0.8474, -0.6643, -0.4939,
         -0.3683, -0.5238, -0.0574, -0.3134, -0.1062, -0.1632,  0.2884, -0.7151,
         -0.5706, -0.4397, -0.0706, -0.3791, -0.1194, -0.6536, -0.1711, -0.2240,
         -0.5716, -0.8022, -0.7476, -0.5880, -0.2765, -0.6157]),
 tensor(0.))

In [18]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [29]:
import torch.nn as nn

class MySimpleNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self,features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [30]:
lr = 0.1
epochs = 25

In [31]:
model = MySimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(),lr)

loss_function = nn.BCELoss()

In [32]:
for epoch in range(epochs):
  for batch_features,batch_labels in train_loader:
    y_pred = model(batch_features)

    loss = loss_function(y_pred,batch_labels.view(-1,1))

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(f'epoch:{epoch+1}, Loss:{loss.item()}')

epoch:1, Loss:0.721465528011322
epoch:1, Loss:0.525232195854187
epoch:1, Loss:0.5144455432891846
epoch:1, Loss:0.36427128314971924
epoch:1, Loss:0.3554893136024475
epoch:1, Loss:0.27002614736557007
epoch:1, Loss:0.2908709645271301
epoch:1, Loss:0.2423529177904129
epoch:1, Loss:0.2824224531650543
epoch:1, Loss:0.242044597864151
epoch:1, Loss:0.1757591962814331
epoch:1, Loss:0.3694725036621094
epoch:1, Loss:0.20963270962238312
epoch:1, Loss:0.19589941203594208
epoch:1, Loss:0.21387319266796112
epoch:2, Loss:0.22326919436454773
epoch:2, Loss:0.21392127871513367
epoch:2, Loss:0.16604720056056976
epoch:2, Loss:0.20251429080963135
epoch:2, Loss:0.15891849994659424
epoch:2, Loss:0.1304291933774948
epoch:2, Loss:0.29273736476898193
epoch:2, Loss:0.1227528378367424
epoch:2, Loss:0.11608222872018814
epoch:2, Loss:0.16116729378700256
epoch:2, Loss:0.16836535930633545
epoch:2, Loss:0.16492904722690582
epoch:2, Loss:0.17077328264713287
epoch:2, Loss:0.10149042308330536
epoch:2, Loss:0.1964005678892

In [33]:
# Model evaluation using test_loader
model.eval()  # Set the model to evaluation mode
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        # Forward pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()  # Convert probabilities to binary predictions

        # Calculate accuracy for the current batch
        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()
        accuracy_list.append(batch_accuracy)

# Calculate overall accuracy
overall_accuracy = sum(accuracy_list) / len(accuracy_list)
print(f'Accuracy: {overall_accuracy:.4f}')


Accuracy: 0.9062
